# Incident Response Capstone

**Level:** Advanced · **Time:** 90 min

In this capstone, we simulate the 3 phases of a production incident response agent.

We will cover:
1. **Evidence Gathering:** Using Read-Only tools to build a timeline.
2. **Impact Synthesis:** Quantifying the blast radius and SLA exposure.
3. **Mitigation Proposal:** Generating a rollback plan and awaiting Human-in-the-Loop approval.

---
## Phase 1: Evidence Gathering (Read-Only)

The agent uses mock Datadog and GitHub APIs to pull facts and build a chronological timeline, avoiding hallucinated assumptions.

In [ ]:
def query_datadog_metrics():
    return "[08:55] Alert: Checkout conversion dropped 31%."

def query_recent_deployments():
    return "[08:49] GitHub Actions: Checkout UI v2.1 deployed successfully."

def evidence_agent():
    print("🔍 [Agent] Starting Evidence Gathering Phase...")
    print("  -> Querying Telemetry...")
    metric = query_datadog_metrics()
    print("  -> Querying CI/CD...")
    deploy = query_recent_deployments()
    
    timeline = f"\nEvidence Timeline:\n1. {deploy}\n2. {metric}"
    print(timeline)
    return timeline

timeline = evidence_agent()


---
## Phase 2: Impact Synthesis (Blast Radius)

The agent queries the tenant database to identify who is failing and calculates the potential SLA penalty.

In [ ]:
def query_affected_tenants():
    # Mock database query
    return [{"tenant": "Acme Corp", "tier": "Enterprise"}, {"tenant": "Globex", "tier": "Standard"}]

def calculate_sla_exposure(tenants):
    exposure = 0
    for t in tenants:
        if t['tier'] == 'Enterprise':
            exposure += 50000  # $50k penalty for Enterprise
    return exposure

def impact_agent():
    print("\n📊 [Agent] Starting Impact Synthesis Phase...")
    print("  -> Querying failed requests by tenant...")
    affected = query_affected_tenants()
    
    print("  -> Calculating SLA exposure...")
    exposure = calculate_sla_exposure(affected)
    
    brief = f"\nIncident Brief:\n- Affected Accounts: {len(affected)}\n- Enterprise SLA Exposure: ${exposure}"
    print(brief)
    return brief

brief = impact_agent()


---
## Phase 3: Mitigation Proposal (Human-in-the-Loop)

The agent drafts a mitigation payload and stops. It routes the proposal to a human for approval. It does NOT execute the payload autonomously.

In [ ]:
import hashlib
import time

def propose_mitigation(timeline, brief):
    print("\n🛡️ [Agent] Drafting Mitigation Proposal...")
    
    payload = {
        "action": "revert_deployment",
        "target": "Checkout UI v2.1",
        "idempotency_key": hashlib.md5(str(time.time()).encode()).hexdigest(),
        "justification": "Deployment directly preceded 31% conversion drop."
    }
    
    print(f"  -> Proposal Payload: {payload}")
    print("  -> 🛑 STOPPING EXECUTION. Routing to Human On-Call (PagerDuty).")
    return payload

def human_approval_webhook(payload, approved):
    print(f"\n[Webhook] Received Human Input: Approved={approved}")
    if approved:
        print(f"✅ [Orchestrator] Executing payload idempotently: {payload['action']} on {payload['target']}")
    else:
        print("❌ [Orchestrator] Mitigation rejected. Agent must find another solution.")

payload = propose_mitigation(timeline, brief)

# Simulate the human waking up and clicking "Approve" 2 minutes later
print("...waiting for human...")
time.sleep(1) 
human_approval_webhook(payload, approved=True)
